# Notebook 1: English pipeline (tuning, main results table, figures)
Run top to bottom on **Google Colab** (use a GPU runtime only if the LaBSE embedding files are missing).
Everything is saved to `ML_Checkpoints` on Drive. The last cell saves the artifacts Notebook 2 needs.

In [ ]:
!pip -q install sentence-transformers

import os, re, time, datetime, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             classification_report, ConfusionMatrixDisplay)

drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/ML_Checkpoints"
os.makedirs(CKPT, exist_ok=True)
SEED = 42

## 1. Data loading and cleaning (same as the earlier milestones)

In [ ]:
base_url = "https://huggingface.co/datasets/google-research-datasets/go_emotions/resolve/main/simplified/"
train_df = pd.read_parquet(base_url + "train-00000-of-00001.parquet")
val_df   = pd.read_parquet(base_url + "validation-00000-of-00001.parquet")
test_df  = pd.read_parquet(base_url + "test-00000-of-00001.parquet")

LABEL_NAMES = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]
GROUP_MAP = {
    "admiration":"positive","amusement":"positive","approval":"positive",
    "caring":"positive","desire":"positive","excitement":"positive",
    "gratitude":"positive","joy":"positive","love":"positive",
    "optimism":"positive","pride":"positive","relief":"positive",
    "anger":"negative","annoyance":"negative","disappointment":"negative",
    "disapproval":"negative","disgust":"negative","embarrassment":"negative",
    "fear":"negative","grief":"negative","nervousness":"negative",
    "remorse":"negative","sadness":"negative",
    "confusion":"ambiguous","curiosity":"ambiguous",
    "realization":"ambiguous","surprise":"ambiguous",
    "neutral":"neutral"
}

def labels_to_groups_all(label_ids):
    names = [LABEL_NAMES[i] for i in label_ids]
    groups = [GROUP_MAP[n] for n in names]
    return list(dict.fromkeys(groups))

def process_split(df):
    """Keep rows whose labels collapse to ONE group; set the rest aside as conflicts."""
    df = df.copy()
    df["all_groups"] = df["labels"].apply(labels_to_groups_all)
    df["num_distinct_groups"] = df["all_groups"].apply(len)
    clean = df[df["num_distinct_groups"] == 1].copy()
    clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
    conflict = df[df["num_distinct_groups"] > 1].copy()
    return clean, conflict

def clean_text(text):
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text)   # remove placeholder tokens
    text = re.sub(r"http\S+", "", text)                  # remove links
    text = re.sub(r"\s+", " ", text).strip()             # collapse whitespace
    return text

def prepare(df):
    clean, conflict = process_split(df)
    clean["clean_text"] = clean["text"].apply(clean_text)
    clean = clean[clean["clean_text"].str.strip() != ""]
    return clean, conflict

train_clean, train_conflict = prepare(train_df)
val_clean,   val_conflict   = prepare(val_df)
test_clean,  test_conflict  = prepare(test_df)

# Remove training rows whose cleaned text also appears in validation or test (leakage)
leak_texts = set(val_clean["clean_text"]) | set(test_clean["clean_text"])
n_leak = int(train_clean["clean_text"].isin(leak_texts).sum())
train_clean = train_clean[~train_clean["clean_text"].isin(leak_texts)]

print(f"Training rows removed for overlapping with validation/test: {n_leak}")
print("Final sizes  train/val/test:", len(train_clean), len(val_clean), len(test_clean))
print(train_clean["group_label"].value_counts())

## 2. Features: TF-IDF and frozen LaBSE embeddings (cached on Drive)

In [ ]:
X_tr, y_tr = train_clean["clean_text"], train_clean["group_label"]
X_va, y_va = val_clean["clean_text"],   val_clean["group_label"]
X_te, y_te = test_clean["clean_text"],  test_clean["group_label"]

# TF-IDF: fit on TRAIN only
tfidf = TfidfVectorizer(max_features=10000)
Xtr_t = tfidf.fit_transform(X_tr)
Xva_t = tfidf.transform(X_va)
Xte_t = tfidf.transform(X_te)

# LaBSE: load cached embeddings if they exist, otherwise compute (slow without a GPU)
labse = None
def get_embeddings(texts, name):
    global labse
    path = f"{CKPT}/labse_{name}.npy"
    if os.path.exists(path):
        E = np.load(path)
        print("Loaded", name, E.shape)
        return E
    if labse is None:
        from sentence_transformers import SentenceTransformer
        labse = SentenceTransformer("sentence-transformers/LaBSE")
    E = labse.encode(list(texts), batch_size=64, show_progress_bar=True)
    np.save(path, E)
    print("Computed and saved", name, E.shape)
    return E

t0 = time.time()
Etr = get_embeddings(X_tr, "train")
Eva = get_embeddings(X_va, "val")
Ete = get_embeddings(X_te, "test")
embed_time = time.time() - t0   # tiny if loaded from cache; the original embedding run took about 66 minutes

assert len(Etr) == len(X_tr) and len(Eva) == len(X_va) and len(Ete) == len(X_te), \
    "Cached embeddings do not match the cleaned data. Delete the labse_*.npy files and rerun to recompute."

## 3. Helpers: bootstrap uncertainty and the experiment log

In [ ]:
def bootstrap_f1(y_true, y_pred, n=1000, seed=SEED):
    """Resample the test set to estimate uncertainty in macro-F1."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = np.array([
        f1_score(y_true[i], y_pred[i], average="macro")
        for i in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n))
    ])
    return scores.mean(), scores.std(), np.percentile(scores, [2.5, 97.5])

def paired_diff(y_true, pred_a, pred_b, n=1000, seed=SEED):
    """95% interval for macro-F1(A) - macro-F1(B) on the same resamples (excludes 0 = meaningful gap)."""
    rng = np.random.default_rng(seed)
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        diffs.append(f1_score(y_true[i], pred_a[i], average="macro")
                     - f1_score(y_true[i], pred_b[i], average="macro"))
    return np.percentile(diffs, [2.5, 97.5])

log = []   # experiment log: one row per run
def log_run(stage, model, params, val_f1, test_f1, seconds, note=""):
    log.append({"run_id": f"E{len(log)+1:03d}", "date": datetime.date.today().isoformat(),
                "stage": stage, "model": model, "hyperparameters": str(params), "seed": SEED,
                "data_split": "GoEmotions official split, 4 groups",
                "val_macro_f1": val_f1, "test_macro_f1": test_f1,
                "runtime_s": round(seconds, 2), "observations": note})

## 4. Tuning: same six-configuration grid per model, chosen on VALIDATION only

In [ ]:
def tune(name, make_model, grid, Xtr, Xva):
    best = None
    for params in grid:
        t0 = time.time()
        m = make_model(**params).fit(Xtr, y_tr)
        val_f1 = f1_score(y_va, m.predict(Xva), average="macro")
        log_run("tuning", name, params, round(val_f1, 4), "-", time.time() - t0)
        if best is None or val_f1 > best[0]:
            best = (val_f1, params)
    return best[1]

make_lr = lambda C: LogisticRegression(C=C, max_iter=1000, random_state=SEED)
nb_grid = [{"alpha": a} for a in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]]
lr_grid = [{"C": c} for c in [0.03, 0.1, 0.3, 1, 3, 10]]

best_nb    = tune("TF-IDF + Naive Bayes",         lambda alpha: MultinomialNB(alpha=alpha), nb_grid, Xtr_t, Xva_t)
best_lr    = tune("TF-IDF + Logistic Regression", make_lr, lr_grid, Xtr_t, Xva_t)
best_labse = tune("LaBSE + Logistic Regression",  make_lr, lr_grid, Etr,   Eva)
print("Selected:", best_nb, best_lr, best_labse)
display(pd.DataFrame(log)[["run_id", "model", "hyperparameters", "val_macro_f1", "runtime_s"]])

## 5. Final fit and main results table (the test set is scored once, here)

In [ ]:
configs = {
    "Naive benchmark (majority class)": (DummyClassifier(strategy="most_frequent"), Xtr_t, Xva_t, Xte_t, "-"),
    "TF-IDF + Naive Bayes":             (MultinomialNB(**best_nb), Xtr_t, Xva_t, Xte_t, best_nb),
    "TF-IDF + Logistic Regression":     (make_lr(**best_lr), Xtr_t, Xva_t, Xte_t, best_lr),
    "LaBSE + Logistic Regression":      (make_lr(**best_labse), Etr, Eva, Ete, best_labse),
}

test_preds, rows = {}, []
for name, (model, Xtr, Xva, Xte, params) in configs.items():
    t0 = time.time(); model.fit(Xtr, y_tr); fit_s = time.time() - t0
    p_va, p_te = model.predict(Xva), model.predict(Xte)
    test_preds[name] = p_te
    mean, sd, (lo, hi) = bootstrap_f1(y_te, p_te)
    amb = f1_score(y_te, p_te, labels=["ambiguous"], average=None)[0]
    val_f1, test_f1 = f1_score(y_va, p_va, average="macro"), f1_score(y_te, p_te, average="macro")
    log_run("final", name, params, round(val_f1, 4), round(test_f1, 4), fit_s,
            "test set scored once; selection used validation only")
    rows.append({
        "Model": name, "Tuned setting": params,
        "Val macro-F1": round(val_f1, 4),
        "Test macro-F1": round(test_f1, 4),
        "Test 95% CI": f"[{lo:.3f}, {hi:.3f}]",
        "Macro precision": round(precision_score(y_te, p_te, average="macro", zero_division=0), 4),
        "Macro recall": round(recall_score(y_te, p_te, average="macro", zero_division=0), 4),
        "Accuracy": round(accuracy_score(y_te, p_te), 4),
        "Ambiguous F1": round(amb, 4),
        "Fit time (s)": round(fit_s, 2),
    })

main_table = pd.DataFrame(rows)
main_table.to_csv(f"{CKPT}/main_results_table.csv", index=False)
pd.DataFrame(log).to_csv(f"{CKPT}/experiment_log.csv", index=False)
display(main_table)
print(f"LaBSE embedding time is NOT in the fit time above (measured this session: {embed_time:.1f} s; original run about 66 minutes).")

ci_lr_nb = paired_diff(y_te, test_preds["TF-IDF + Logistic Regression"], test_preds["TF-IDF + Naive Bayes"])
ci_lb_lr = paired_diff(y_te, test_preds["LaBSE + Logistic Regression"], test_preds["TF-IDF + Logistic Regression"])
print("LR - NB    95% interval:", ci_lr_nb.round(4))
print("LaBSE - LR 95% interval:", ci_lb_lr.round(4))
pd.DataFrame({"comparison": ["LogReg minus NaiveBayes", "LaBSE minus LogReg"],
              "ci_low": [ci_lr_nb[0], ci_lb_lr[0]], "ci_high": [ci_lr_nb[1], ci_lb_lr[1]]}
             ).to_csv(f"{CKPT}/paired_differences.csv", index=False)

## 6. Figures: main results bar chart and confusion matrices

In [ ]:
names  = ["Naive benchmark (majority class)", "TF-IDF + Naive Bayes",
          "TF-IDF + Logistic Regression", "LaBSE + Logistic Regression"]
labels = ["Majority\nclass", "TF-IDF +\nNaive Bayes", "TF-IDF +\nLogistic Reg.", "LaBSE +\nLogistic Reg."]

pts, lows, highs = [], [], []
for n in names:
    p = f1_score(y_te, test_preds[n], average="macro")
    _, _, (lo, hi) = bootstrap_f1(y_te, test_preds[n])
    pts.append(p); lows.append(max(p - lo, 0)); highs.append(max(hi - p, 0))

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels, pts, yerr=[lows, highs], capsize=6, color=["#9aa5a8", "#6AA4C8", "#1A9988", "#1C3678"])
for b, p in zip(bars, pts):
    ax.text(b.get_x() + b.get_width() / 2, p + 0.025, f"{p:.3f}", ha="center", fontsize=10)
ax.set_ylabel("Test macro-F1"); ax.set_ylim(0, 0.75); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CKPT}/fig_main_results.png", dpi=200)
plt.show()

LABELS = ["positive", "negative", "ambiguous", "neutral"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, key) in zip(axes, [("Naive Bayes", "TF-IDF + Naive Bayes"),
                                   ("Logistic Regression", "TF-IDF + Logistic Regression"),
                                   ("LaBSE + Logistic Regression", "LaBSE + Logistic Regression")]):
    ConfusionMatrixDisplay.from_predictions(y_te, test_preds[key], labels=LABELS,
        normalize="true", values_format=".2f", cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(f"{CKPT}/fig_confusion_matrices.png", dpi=200)
plt.show()

## 7. Per-class reports (for the error analysis)

In [ ]:
for key in ["TF-IDF + Naive Bayes", "TF-IDF + Logistic Regression", "LaBSE + Logistic Regression"]:
    print(key)
    print(classification_report(y_te, test_preds[key], digits=3))

## 8. Save the English artifacts that Notebook 2 (French test) needs

In [ ]:
artifacts = {
    "tfidf": tfidf,
    "models": {k: configs[k][0] for k in ["TF-IDF + Naive Bayes", "TF-IDF + Logistic Regression", "LaBSE + Logistic Regression"]},
    "test_ids": test_clean["id"].astype(str).to_numpy(),
    "test_labels": test_clean["group_label"].to_numpy(),
    "test_preds": {k: np.asarray(test_preds[k]) for k in ["TF-IDF + Naive Bayes", "TF-IDF + Logistic Regression", "LaBSE + Logistic Regression"]},
}
joblib.dump(artifacts, f"{CKPT}/english_crosslingual_artifacts.joblib")
print("Saved. Files in", CKPT, ":", sorted(os.listdir(CKPT)))

## 9. Optional: learning curve with the TUNED settings (sensitivity to training-set size)

In [ ]:
fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
rng = np.random.default_rng(SEED)
curve = {"Naive Bayes": [], "Logistic Regression": [], "LaBSE + Logistic Regression": []}

for frac in fractions:
    idx = rng.choice(len(X_tr), int(frac * len(X_tr)), replace=False)
    ys = y_tr.iloc[idx]
    vec = TfidfVectorizer(max_features=10000)
    Xs_t, Xv_t = vec.fit_transform(X_tr.iloc[idx]), vec.transform(X_va)
    curve["Naive Bayes"].append(f1_score(y_va, MultinomialNB(**best_nb).fit(Xs_t, ys).predict(Xv_t), average="macro"))
    curve["Logistic Regression"].append(f1_score(y_va, make_lr(**best_lr).fit(Xs_t, ys).predict(Xv_t), average="macro"))
    curve["LaBSE + Logistic Regression"].append(f1_score(y_va, make_lr(**best_labse).fit(Etr[idx], ys).predict(Eva), average="macro"))

for name, scores in curve.items():
    plt.plot([f * len(X_tr) for f in fractions], scores, marker="o", label=name)
plt.xlabel("Training examples"); plt.ylabel("Validation macro-F1")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{CKPT}/fig_learning_curve.png", dpi=200)
plt.show()
print({k: [round(s, 3) for s in v] for k, v in curve.items()})